# 01 - Industry Data Generation for Marketing Mix Modeling

## Objective

Generate a realistic synthetic dataset for a retail/e-commerce business that will be used throughout the MMM project.

### Dataset characteristics

- 5 years of weekly data (260 weeks)
- Marketing spend across multiple channels
- Trend and seasonality
- Holidays and festivals
- Competitor activity
- Pricing
- Weather
- Sales generated from multiple business drivers

> This notebook intentionally creates *synthetic but realistic* data for learning and experimentation.


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

np.random.seed(42)

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

weeks = pd.date_range("2021-01-04", periods=260, freq="W-MON")
df = pd.DataFrame({"Week": weeks})


## Generate Media Spend

In [ ]:

channels = [
    "Google_Search","Google_Display","Meta","Instagram",
    "YouTube","TV","Radio","Influencer","Affiliate","Email"
]

for c in channels:
    df[c] = np.random.randint(20000,200000,len(df))

df.head()


## Generate Business Drivers

In [ ]:

df["Discount"] = np.random.choice([0,5,10,15,20], len(df))
df["Price"] = np.random.normal(1000,40,len(df))
df["Temperature"] = 28 + 8*np.sin(np.arange(len(df))/52*2*np.pi)
df["Competitor_Spend"] = np.random.randint(50000,250000,len(df))

df["Holiday"] = 0
df.loc[df["Week"].dt.month.isin([10,11,12]),"Holiday"]=1

trend = np.linspace(0,500000,len(df))
seasonality = 200000*np.sin(np.arange(len(df))/52*2*np.pi)


## Generate Sales

In [ ]:

media_effect = (
    df["Google_Search"]*2.1 +
    df["Meta"]*1.8 +
    df["TV"]*1.4 +
    df["YouTube"]*1.2
)

noise = np.random.normal(0,120000,len(df))

df["Sales"] = (
    2500000
    + trend
    + seasonality
    + media_effect
    + df["Discount"]*25000
    - df["Competitor_Spend"]*0.3
    + df["Holiday"]*350000
    + noise
).astype(int)

df["Revenue"] = df["Sales"]
df["Orders"] = (df["Sales"]/df["Price"]).astype(int)

df[["Week","Sales","Revenue","Orders"]].head()


## Visualize Sales

In [ ]:

plt.figure(figsize=(14,5))
plt.plot(df["Week"],df["Sales"])
plt.title("Weekly Sales")
plt.xlabel("Week")
plt.ylabel("Sales")
plt.grid(True)
plt.show()


## Save Dataset

In [ ]:

output = RAW / "marketing_mix_data.csv"
df.to_csv(output,index=False)

print("Saved to:", output)


# Business Interpretation

Notice that sales are influenced by:

- Marketing spend
- Trend
- Seasonality
- Holiday uplift
- Competitor spend
- Random business variation

This is still a simplified model. Later notebooks will introduce:

- Geometric adstock
- Weibull adstock
- Hill saturation
- Price elasticity
- Multiple regions
- Multiple SKUs
- Campaign calendars
- Bayesian data generation

## Interview Questions

1. Why do we use synthetic data?
2. Why include seasonality?
3. Why simulate competitor spend?
4. Why add random noise?
5. Why avoid perfectly linear relationships?
